In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import cv2
import mediapipe as mp

# Enable mixed precision for faster training on compatible GPUs
mixed_precision.set_global_policy('mixed_float16')

# ==================== DATA LOADING ====================
train_dir = "FER-2013/train"
test_dir = "FER-2013/test"

mood_map = {
    "angry": "angry",
    "disgust": "neutral",
    "fear": "neutral",
    "surprise": "neutral",
    "happy": "happy",
    "sad": "sad",
    "neutral": "neutral"
}

final_moods = ["angry", "happy", "sad", "neutral"]
num_classes = len(final_moods)
mood_to_index = {m: i for i, m in enumerate(final_moods)}

def map_original_to_final(orig_class_name):
    final = mood_map.get(orig_class_name, "neutral")
    return mood_to_index[final]

IMG_SIZE = (48, 48)

def load_dataset(directory, batch_size=64, shuffle=True, augment=False):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        directory,
        labels="inferred",
        label_mode="int",
        image_size=IMG_SIZE,
        color_mode="grayscale",
        batch_size=batch_size,
        shuffle=shuffle
    )
    
    class_names = ds.class_names
    orig_to_new = np.array([map_original_to_final(orig) for orig in class_names], dtype=np.int32)
    
    def map_labels(images, labels):
        new_labels = tf.gather(orig_to_new, labels)
        return images, new_labels
    
    ds = ds.map(map_labels, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Data augmentation for training
    if augment:
        data_augmentation = tf.keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
            layers.RandomContrast(0.1),
        ])
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), 
                    num_parallel_calls=tf.data.AUTOTUNE)
    
    return ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# Load datasets
batch_size = 128  # Increased for better GPU utilization
train_ds_full = load_dataset(train_dir, batch_size=batch_size, shuffle=True, augment=True)
test_ds = load_dataset(test_dir, batch_size=batch_size, shuffle=False)

# Split validation from training
train_size = tf.data.experimental.cardinality(train_ds_full).numpy()
val_size = int(train_size * 0.1)
val_ds = train_ds_full.take(val_size)
train_ds = train_ds_full.skip(val_size)

# ==================== MODEL BUILDING ====================
def build_optimized_model(input_shape=(48, 48, 1), num_classes=4):
    """Optimized CNN with residual connections and efficient architecture"""
    inputs = layers.Input(shape=input_shape)
    
    # Initial conv block
    x = layers.Conv2D(32, (3, 3), padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    # Block 1 with residual
    residual = layers.Conv2D(64, (1, 1), strides=(2, 2), padding="same")(x)
    x = layers.Conv2D(64, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, (3, 3), strides=(2, 2), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.25)(x)
    
    # Block 2 with residual
    residual = layers.Conv2D(128, (1, 1), strides=(2, 2), padding="same")(x)
    x = layers.Conv2D(128, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(128, (3, 3), strides=(2, 2), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.3)(x)
    
    # Global pooling instead of flatten
    x = layers.GlobalAveragePooling2D()(x)
    
    # Dense layers
    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.5)(x)
    
    outputs = layers.Dense(num_classes, activation="softmax", dtype='float32')(x)
    
    model = models.Model(inputs, outputs)
    
    # Use AdamW optimizer with cosine decay
    initial_lr = 1e-3
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_lr, decay_steps=1000
    )
    optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-5)
    
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )
    return model

model = build_optimized_model()
model.summary()

# ==================== TRAINING ====================
callbacks = [
    ModelCheckpoint("mood_model_best.h5", monitor="val_accuracy",
                   save_best_only=True, mode="max", verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=10, 
                 restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, 
                     min_lr=1e-7, verbose=1)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
    verbose=1
)

# ==================== EVALUATION ====================
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest accuracy: {test_acc:.4f}")

# ==================== REAL-TIME DETECTION ====================
class OptimizedMoodDetector:
    def __init__(self, model_path="mood_model_best.h5"):
        self.model = tf.keras.models.load_model(model_path)
        self.final_moods = ["angry", "happy", "sad", "neutral"]
        
        # MediaPipe setup
        self.mp_face_detection = mp.solutions.face_detection
        self.face_detector = self.mp_face_detection.FaceDetection(
            min_detection_confidence=0.7,
            model_selection=1  # 1 for full range detection
        )
        
        # Frame skip for performance
        self.frame_count = 0
        self.skip_frames = 2  # Process every 3rd frame
        self.last_prediction = "neutral"
        
    def preprocess_face(self, gray_roi):
        img = cv2.resize(gray_roi, (48, 48), interpolation=cv2.INTER_AREA)
        img = img.astype("float32") / 255.0
        img = np.expand_dims(img, axis=[0, -1])
        return img
    
    def run(self):
        cap = cv2.VideoCapture(0)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        cap.set(cv2.CAP_PROP_FPS, 30)
        
        print("Press 'q' to quit")
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            self.frame_count += 1
            
            # Skip frames for better performance
            if self.frame_count % self.skip_frames != 0:
                cv2.putText(frame, f"Mood: {self.last_prediction}", 
                          (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 
                          1, (0, 255, 0), 2)
                cv2.imshow("Mood Detection", frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
                continue
            
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            det_res = self.face_detector.process(img_rgb)
            
            if det_res.detections:
                for det in det_res.detections:
                    bbox = det.location_data.relative_bounding_box
                    h, w = frame.shape[:2]
                    x1 = max(0, int(bbox.xmin * w))
                    y1 = max(0, int(bbox.ymin * h))
                    x2 = min(w, x1 + int(bbox.width * w))
                    y2 = min(h, y1 + int(bbox.height * h))
                    
                    if x2 <= x1 or y2 <= y1:
                        continue
                    
                    face = frame[y1:y2, x1:x2]
                    if face.size == 0:
                        continue
                    
                    face_gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
                    inp = self.preprocess_face(face_gray)
                    preds = self.model.predict(inp, verbose=0)
                    class_id = np.argmax(preds[0])
                    confidence = preds[0][class_id]
                    mood = self.final_moods[class_id]
                    self.last_prediction = mood
                    
                    # Draw results
                    color = (0, 255, 0) if confidence > 0.6 else (0, 165, 255)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    text = f"{mood} ({confidence:.2f})"
                    cv2.putText(frame, text, (x1, y1 - 10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            
            cv2.imshow("Mood Detection", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()

# Run detection
if __name__ == "__main__":
    detector = OptimizedMoodDetector()
    detector.run()

Found 28709 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 48, 48, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast (Cast)         │ (None, 48, 48, 1) │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 48, 48,    │        320 │ cast[0][0]        │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 48, 48,    │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 48, 48,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 48, 48,    │     18,496 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 48, 48,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 48, 48,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 24, 24,    │     36,928 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 24, 24,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 24, 24,    │      2,112 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 24, 24,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 24, 24,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 24, 24,    │          0 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 24, 24,    │     73,856 │ dropout[0][0]     │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 24, 24,    │        512 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 24, 24,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                 

 Total params: 324,356 (1.24 MB)

 Trainable params: 323,012 (1.23 MB)

 Non-trainable params: 1,344 (5.25 KB)

Epoch 1/50
